# exp_10c — Experiment 10C: Variational Autoencoder (VAE) for MNIST

Source-based notebook converted from the uploaded experiment code document.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from tensorflow.keras.datasets import mnist

(X_train, _), (X_test, _) = mnist.load_data()

X_train = X_train.astype("float32") / 255.
X_test = X_test.astype("float32") / 255.

X_train = X_train.reshape(-1, 784)
X_test = X_test.reshape(-1, 784)

In [ ]:
input_dim = 784
hidden_dim = 256
latent_dim = 2

In [ ]:
inputs = layers.Input(shape=(input_dim,))

x = layers.Dense(hidden_dim, activation="relu")(inputs)

z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)

In [ ]:
class Sampling(layers.Layer):

    def call(self, inputs):

        z_mean, z_log_var = inputs

        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]

        epsilon = tf.random.normal(shape=(batch, dim))

        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [ ]:
z = Sampling()([z_mean, z_log_var])

encoder = Model(inputs, [z_mean, z_log_var, z])
encoder.summary()

In [ ]:
latent_inputs = layers.Input(shape=(latent_dim,))

x = layers.Dense(hidden_dim, activation="relu")(latent_inputs)

outputs = layers.Dense(
    input_dim,
    activation="sigmoid"
)(x)

decoder = Model(latent_inputs, outputs)

decoder.summary()

In [ ]:
class VAE(Model):

    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):

        with tf.GradientTape() as tape:

            # Encode
            z_mean, z_log_var, z = self.encoder(data)

            # Decode
            reconstruction = self.decoder(z)

            # Reconstruction Loss
            reconstruction_loss = tf.reduce_mean(
                tf.keras.losses.binary_crossentropy(
                    data,
                    reconstruction
                )
            ) * input_dim

            # KL Divergence Loss
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var
                - tf.square(z_mean)
                - tf.exp(z_log_var)
            )

            # Total Loss
            total_loss = reconstruction_loss + kl_loss

        grads = tape.gradient(
            total_loss,
            self.trainable_weights
        )

        self.optimizer.apply_gradients(
            zip(grads, self.trainable_weights)
        )

        return {
            "loss": total_loss,
            "reconstruction_loss": reconstruction_loss,
            "kl_loss": kl_loss
        }

In [ ]:
vae = VAE(encoder, decoder)

vae.compile(optimizer="adam")

history = vae.fit(
    X_train,
    epochs=10,
    batch_size=256
)

In [ ]:
random_vector = tf.random.normal(shape=(10, latent_dim))

generated = decoder.predict(random_vector)

In [ ]:
plt.figure(figsize=(15,3))

for i in range(10):
    plt.subplot(1,10,i+1)
    plt.imshow(generated[i].reshape(28,28), cmap="gray")
    plt.axis("off")

plt.show()

In [ ]:
z_mean, _, _ = encoder.predict(X_test)

plt.figure(figsize=(8,6))
plt.scatter(z_mean[:,0], z_mean[:,1], s=3)
plt.xlabel("Latent Dimension 1")
plt.ylabel("Latent Dimension 2")
plt.title("VAE Latent Space")
plt.show()

In [ ]:
import pandas as pd

comparison = pd.DataFrame({

    "Model":[
        "Basic Autoencoder",
        "Denoising Autoencoder",
        "Variational Autoencoder"
    ],

    "Purpose":[
        "Compression & Feature Extraction",
        "Image Denoising",
        "Image Generation"
    ],

    "Latent Space":[
        "Deterministic",
        "Deterministic",
        "Probabilistic"
    ]

})

comparison